# ML 08c — Random Forest Regression

This notebook trains and validates Random Forest Regression using the frozen chronological split. It is an independent experiment with the same predictors, folds, preprocessing rules and metrics used by every other ML notebook. The 2024 final-test target remains sealed.

## Experiment safeguards

- Preprocessing is fitted inside the pipeline separately for every expanding-window fold.
- Unknown validation categories are ignored safely by one-hot encoding.
- Model configuration is fixed before 2023 validation scoring.
- Reports log-scale and kilogram-scale metrics.
- Never fits, predicts or scores the 2024 test partition.

## 1. Prepare the Colab environment

This cell checks the required Python packages and installs only those that are missing from the current Colab runtime.

In [ ]:
import importlib.util, subprocess, sys
packages={'pandas':'pandas>=2.0,<3','numpy':'numpy>=1.24,<3','pyarrow':'pyarrow>=14,<22','matplotlib':'matplotlib>=3.7,<4','sklearn':'scikit-learn>=1.3,<2','joblib':'joblib>=1.3,<2'}
missing=[spec for module,spec in packages.items() if importlib.util.find_spec(module) is None]
if missing: subprocess.check_call([sys.executable,'-m','pip','install',*missing])
print('Random Forest Regression dependencies are available.')

## 2. Import preprocessing, modelling and evaluation tools

The experiment uses a fixed random seed so that model fitting, permutation importance and saved evidence are reproducible.

In [ ]:
from __future__ import annotations
import hashlib, json, os, sys, time
from pathlib import Path
import joblib
import matplotlib
if 'IPython' not in sys.modules: matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
try:
    from IPython.display import display
except ImportError:
    display=print
plt.style.use('seaborn-v0_8-whitegrid')
EXPERIMENT='random_forest'; RANDOM_STATE=42

## 3. Load the frozen split and feature roles

The prepared feature table, feature dictionary and expanding-window folds are loaded from Drive. Only training and 2023 validation rows are exposed; the 2024 test target remains sealed.

In [ ]:
def find_component_root(start:Path)->Path:
    for anchor in [start.resolve(),*start.resolve().parents]:
        for candidate in [anchor,anchor/'component - 3',anchor/'J26-DS-350'/'component - 3']:
            if (candidate/'Notebooks').is_dir() and (candidate/'src').is_dir(): return candidate.resolve()
    raise FileNotFoundError('Component 3 repository root was not found.')
IN_COLAB='google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive'); ROOT=Path('/content/drive/MyDrive/FishGuard-C3')
else: ROOT=find_component_root(Path.cwd())
DATA_PATH=ROOT/'data'/'processed'/'monthly_model_features_split.parquet'
DICTIONARY_PATH=ROOT/'outputs'/'audit'/'feature_engineering'/'feature_dictionary.csv'
FOLDS_PATH=ROOT/'outputs'/'audit'/'modelling_eda_split'/'expanding_window_cv_folds.csv'
MODEL_DIR=ROOT/'models'; EVIDENCE_DIR=ROOT/'outputs'/'audit'/'model_training'/EXPERIMENT; METRIC_DIR=ROOT/'outputs'/'metrics'/'model_training'/EXPERIMENT; PREDICTION_DIR=ROOT/'outputs'/'predictions'/'model_training'/EXPERIMENT; FIGURE_DIR=ROOT/'outputs'/'figures'/'model_training'/EXPERIMENT
for directory in [MODEL_DIR,EVIDENCE_DIR,METRIC_DIR,PREDICTION_DIR,FIGURE_DIR]: directory.mkdir(parents=True,exist_ok=True)
for path in [DATA_PATH,DICTIONARY_PATH,FOLDS_PATH]:
    if not path.is_file(): raise FileNotFoundError(f'Missing {path}.')
data=pd.read_parquet(DATA_PATH); data['month_start']=pd.to_datetime(data.month_start); dictionary=pd.read_csv(DICTIONARY_PATH); folds=pd.read_csv(FOLDS_PATH)
predictors=dictionary.loc[dictionary.role=='candidate_predictor','column'].tolist(); TARGET='target_log1p_median_catch'
train=data.loc[data.data_split.astype(str)=='train'].copy(); validation=data.loc[data.data_split.astype(str)=='validation'].copy(); test_rows=int((data.data_split.astype(str)=='test').sum())
numeric=[column for column in predictors if pd.api.types.is_numeric_dtype(train[column])]; categorical_features=[column for column in predictors if column not in numeric]
source_sha256=hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
print('Train:',len(train),'Validation:',len(validation),'Sealed test:',test_rows,'Predictors:',len(predictors))

## 4. Build the leakage-safe Random Forest pipeline

Missing values and categorical encoding are learned inside each fit. The forest uses many trees for stable averaging, constrained leaf size and row/feature sampling to reduce overfitting while retaining nonlinear interactions.

In [ ]:
numeric_pipeline=Pipeline([('imputer',SimpleImputer(strategy='median',add_indicator=True))])
categorical_pipeline=Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore',sparse_output=False))])
preprocessor=ColumnTransformer([('numeric',numeric_pipeline,numeric),('categorical',categorical_pipeline,categorical_features)],verbose_feature_names_out=False)
estimator=RandomForestRegressor(n_estimators=800,max_depth=None,min_samples_split=6,min_samples_leaf=3,max_features=.7,max_samples=.85,bootstrap=True,n_jobs=-1,random_state=RANDOM_STATE)
pipeline=Pipeline([('preprocess',preprocessor),('model',estimator)])
configuration={'n_estimators':800,'max_depth':None,'min_samples_split':6,'min_samples_leaf':3,'max_features':0.7,'max_samples':0.85,'bootstrap':True,'random_state':42}
print(configuration)

## 5. Run expanding-window cross-validation

Three chronological folds test whether the model transfers to later periods. Both log-scale and back-transformed kilogram metrics are recorded for a fair comparison with the other models.

In [ ]:
def metrics(y_true_log,y_pred_log):
    y_true_log=np.asarray(y_true_log,float); y_pred_log=np.asarray(y_pred_log,float)
    true_kg=np.clip(np.expm1(y_true_log),0,None); pred_kg=np.clip(np.expm1(np.clip(y_pred_log,0,None)),0,None)
    return {'log_mae':mean_absolute_error(y_true_log,y_pred_log),'log_rmse':mean_squared_error(y_true_log,y_pred_log)**.5,'log_median_ae':median_absolute_error(y_true_log,y_pred_log),'log_r2':r2_score(y_true_log,y_pred_log),'kg_mae':mean_absolute_error(true_kg,pred_kg),'kg_rmse':mean_squared_error(true_kg,pred_kg)**.5,'kg_median_ae':median_absolute_error(true_kg,pred_kg),'kg_bias':float(np.mean(pred_kg-true_kg))}
cv_rows=[]
for fold in folds.itertuples(index=False):
    fold_train=train.loc[train.month_start<=pd.Timestamp(fold.train_end)]; fold_valid=train.loc[(train.month_start>=pd.Timestamp(fold.validation_start))&(train.month_start<=pd.Timestamp(fold.validation_end))]
    fitted=clone(pipeline); started=time.perf_counter(); fitted.fit(fold_train[predictors],fold_train[TARGET]); elapsed=time.perf_counter()-started; prediction=fitted.predict(fold_valid[predictors])
    cv_rows.append({'fold_id':int(fold.fold_id),'model':EXPERIMENT,'train_rows':len(fold_train),'validation_rows':len(fold_valid),'fit_seconds':elapsed,**metrics(fold_valid[TARGET],prediction)})
    print(f'Completed fold {int(fold.fold_id)}')
cv=pd.DataFrame(cv_rows); cv.to_csv(METRIC_DIR/'cv_metrics.csv',index=False)
display(cv)

## 6. Fit on all training data and evaluate 2023

After cross-validation, the unchanged pipeline is fitted on the complete development-training period and evaluated once on the untouched 2023 validation partition.

In [ ]:
started=time.perf_counter(); fitted_pipeline=clone(pipeline).fit(train[predictors],train[TARGET]); fit_seconds=time.perf_counter()-started
validation_prediction=fitted_pipeline.predict(validation[predictors]); validation_result={'model':EXPERIMENT,'fit_seconds':fit_seconds,**metrics(validation[TARGET],validation_prediction)}
validation_metrics=pd.DataFrame([validation_result]); validation_metrics.to_csv(METRIC_DIR/'validation_metrics.csv',index=False)
predictions=validation[['month_start','spatial_cell_id','species_model','gear_model',TARGET,'target_median_catch_weight_kg']].copy().rename(columns={'target_median_catch_weight_kg':'actual_kg'})
predictions['prediction_log']=validation_prediction; predictions['prediction_kg']=np.expm1(predictions.prediction_log.clip(lower=0)); predictions['residual_log']=predictions[TARGET]-predictions.prediction_log
predictions.to_csv(PREDICTION_DIR/'validation_predictions.csv',index=False)
summary=pd.DataFrame([{'model':EXPERIMENT,'cv_log_mae_mean':cv.log_mae.mean(),'cv_log_mae_std':cv.log_mae.std(),'cv_kg_mae_mean':cv.kg_mae.mean(),**validation_result}])
summary.to_csv(METRIC_DIR/'experiment_summary.csv',index=False)
MODEL_PATH=MODEL_DIR/f'{EXPERIMENT}_development_pipeline.joblib'; joblib.dump(fitted_pipeline,MODEL_PATH); model_sha256=hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest()
display(summary)

## 7. Diagnose validation behaviour and feature importance

Residual plots reveal systematic prediction errors, temporal scores show stability across years, and permutation importance estimates which original predictors contribute most on unseen 2023 data.

In [ ]:
importance=permutation_importance(fitted_pipeline,validation[predictors],validation[TARGET],scoring='neg_mean_absolute_error',n_repeats=5,random_state=RANDOM_STATE,n_jobs=-1)
importance_table=pd.DataFrame({'feature':predictors,'importance_mean':importance.importances_mean,'importance_std':importance.importances_std}).sort_values('importance_mean',ascending=False)
importance_table.to_csv(METRIC_DIR/'validation_permutation_importance.csv',index=False)
fig,axes=plt.subplots(1,2,figsize=(14,6))
axes[0].scatter(predictions[TARGET],predictions.prediction_log,alpha=.5,color='#59a14f'); limits=[min(predictions[TARGET].min(),predictions.prediction_log.min()),max(predictions[TARGET].max(),predictions.prediction_log.max())]; axes[0].plot(limits,limits,'--',color='black'); axes[0].set_xlim(limits); axes[0].set_ylim(limits); axes[0].set_xlabel('Actual log1p catch'); axes[0].set_ylabel('Predicted log1p catch'); axes[0].set_title('Random Forest Regression: 2023 validation')
axes[1].scatter(predictions.prediction_log,predictions.residual_log,alpha=.5,color='#59a14f'); axes[1].axhline(0,color='black',linestyle='--'); axes[1].set_xlabel('Predicted log1p catch'); axes[1].set_ylabel('Actual − predicted'); axes[1].set_title('Validation residuals')
fig.tight_layout(); fig.savefig(FIGURE_DIR/'validation_diagnostics.png',dpi=200,bbox_inches='tight'); plt.show()
top=importance_table.head(20).sort_values('importance_mean'); fig,ax=plt.subplots(figsize=(11,8)); ax.barh(top.feature,top.importance_mean,xerr=top.importance_std,color='#59a14f'); ax.axvline(0,color='black',linewidth=.8); ax.set_xlabel('Increase in validation MAE when permuted'); ax.set_title('Random Forest Regression: permutation importance'); fig.tight_layout(); fig.savefig(FIGURE_DIR/'permutation_importance.png',dpi=200,bbox_inches='tight'); plt.show()
fig,ax=plt.subplots(figsize=(8,5)); ax.plot(cv.fold_id,cv.log_mae,marker='o',color='#59a14f'); ax.axhline(validation_result['log_mae'],color='black',linestyle='--',label='2023 validation'); ax.set_xticks(cv.fold_id); ax.set_xlabel('Expanding-window fold'); ax.set_ylabel('Log MAE'); ax.set_title('Random Forest Regression: temporal performance'); ax.legend(); fig.tight_layout(); fig.savefig(FIGURE_DIR/'temporal_performance.png',dpi=200,bbox_inches='tight'); plt.show()
display(importance_table.head(20))

## 8. Verify and document the Random Forest experiment

Final assertions confirm split integrity, target exclusion, finite predictions and artifact integrity before saving the audit findings and reproducibility manifest.

In [ ]:
checks=pd.DataFrame([
 {'check':'test excluded','passed':len(train)+len(validation)+test_rows==len(data),'detail':f'{test_rows} sealed rows'},
 {'check':'target excluded','passed':TARGET not in predictors and not any(x.startswith('target_') for x in predictors),'detail':f'{len(predictors)} predictors'},
 {'check':'pipeline preprocessing','passed':isinstance(fitted_pipeline,Pipeline),'detail':'imputation/encoding inside pipeline'},
 {'check':'three folds','passed':cv.fold_id.nunique()==3,'detail':str(sorted(cv.fold_id.unique()))},
 {'check':'finite validation predictions','passed':np.isfinite(validation_prediction).all(),'detail':f'{len(validation_prediction)} predictions'},
 {'check':'model hash','passed':hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest()==model_sha256,'detail':MODEL_PATH.name}])
assert checks.passed.all(); checks.to_csv(EVIDENCE_DIR/'experiment_checks.csv',index=False)
findings=pd.DataFrame([{'finding':'Model','result':'Random Forest Regression'},{'finding':'Mean expanding-window log MAE','result':f'{cv.log_mae.mean():.4f}'},{'finding':'2023 validation log MAE','result':f"{validation_result['log_mae']:.4f}"},{'finding':'2023 validation kg MAE','result':f"{validation_result['kg_mae']:.2f} kg"},{'finding':'2023 validation kg bias','result':f"{validation_result['kg_bias']:.2f} kg"},{'finding':'Test status','result':'2024 target sealed and unused'}])
findings.to_csv(EVIDENCE_DIR/'experiment_findings.csv',index=False)
manifest={'created_utc':pd.Timestamp.now(tz='UTC').isoformat(),'runtime':'Google Colab' if IN_COLAB else 'Local Windows','experiment':EXPERIMENT,'configuration':configuration,'input':DATA_PATH.name,'input_sha256':source_sha256,'train_rows':len(train),'validation_rows':len(validation),'sealed_test_rows':test_rows,'test_target_used':False,'predictors':predictors,'model_file':MODEL_PATH.name,'model_sha256':model_sha256,'metrics':sorted(p.name for p in METRIC_DIR.glob('*.csv')),'predictions':sorted(p.name for p in PREDICTION_DIR.glob('*.csv')),'figures':sorted(p.name for p in FIGURE_DIR.glob('*.png'))}
with (EVIDENCE_DIR/'experiment_manifest.json').open('w',encoding='utf-8') as f: json.dump(manifest,f,indent=2)
display(checks); display(findings)

## Viva interpretation

Random Forest captures nonlinearities and interactions without assuming a linear response. Minimum leaf size reduces overfitting, while chronological evaluation reveals whether learned patterns transfer forward in time.